In [1]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 11.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [4]:
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI3 = "databricks/phoenix-7b-instruct"
gemma = "google/gemma-2b-it"

In [5]:
message = [{
    "role": "user",
    "content": "Tell me light hearted joke"
}]

In [6]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [7]:
#Tokenization

tokenizer = AutoTokenizer.from_pretrained(gemma)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(message, return_tensors="pt")

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [8]:
model = AutoModelForCausalLM.from_pretrained(gemma, quantization_config=quant_config, device_map="auto")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [9]:
model

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (n

In [10]:
outputs = model.generate(inputs=inputs, max_new_tokens=80)
print(tokenizer.decode(outputs[0]))

<bos><start_of_turn>user
Tell me light hearted joke<end_of_turn>
Why did the scarecrow win an award?

Because he was outstanding in his field!<eos>


In [11]:
def generate(model, messages):
  """Generates and streams a text response from a Hugging Face model.

  This function takes a loaded language model and a conversation history,
  tokenizes the input, and generates a response. The output is streamed
  token-by-token directly to the console. After generation, it cleans
  up memory by deleting variables and emptying the CUDA cache.

  Args:
    model (transformers.PreTrainedModel): The loaded Hugging Face model object
        that has a `.generate()` method.
    messages (List[Dict[str, str]]): A list of dictionaries representing the
        conversation history, formatted for `apply_chat_template`.

  Returns:
      None. The output is printed directly to the console.

  Note:
    This function currently uses a hardcoded tokenizer ('gemma') and message
    variable, which makes it not very reusable. For better design, the
    tokenizer and messages should also be passed as arguments.
  """
  tokenizer = AutoTokenizer.from_pretrained(gemma)
  tokenizer.pad_token = tokenizer.eos_token
  inputs = tokenizer.apply_chat_template(message, return_tensors="pt")
  streamers = TextStreamer(tokenizer)
  outputs = model.generate(inputs=inputs, max_new_tokens=80, streamer=streamers)

  del tokenizer, streamers, model, inputs, outputs
  torch.cuda.empty_cache()


In [14]:
generate(model, message)

<bos><start_of_turn>user
Tell me light hearted joke<end_of_turn>
Why did the scarecrow win an award?

Because he was outstanding in his field!<eos>
